In [24]:
%uv pip install torch triton

Using Python 3.12.6 environment at: /usr/local
Audited 2 packages in 12ms
Note: you may need to restart the kernel to use updated packages.


In [25]:
import torch

import triton
import triton.language as tl

In [26]:
@triton.jit
def sum_k(in_ptr, part_ptr, num_el, BLOCK_SIZE: tl.constexpr):

    pid = tl.program_id(0)

    block_start = pid * BLOCK_SIZE

    offs = block_start + tl.arange(0, BLOCK_SIZE)

    mask = offs < num_el

    values = tl.load(in_ptr+offs, mask=mask, other=0.0)

    block_sum = tl.sum(values, axis=0)

    tl.store(part_ptr+pid, block_sum)

def triton_sum(x):
    assert x.is_cuda
    assert x.dtype == torch.float32
    assert x.ndim == 1
    assert x.is_contiguous()

    if x.numel() == 0:
        return torch.zeros(1, device=x.device, dtype=x.dtype)

    curr = x
    block_size = 1024

    while curr.numel() > 1:
        num_el = curr.numel()

        num_programs = triton.cdiv(
            num_el, block_size
        )

        partial = torch.empty(
            num_programs, device=x.device, dtype=x.dtype
        )

        grid = (num_programs, )

        sum_k[grid](
            curr, partial,
            num_el, BLOCK_SIZE=block_size,
            num_warps=8
        )

        curr = partial

    return curr

In [27]:
@triton.jit
def max_k(in_ptr, part_ptr, num_el, BLOCK_SIZE:tl.constexpr):

    pid = tl.program_id(0)

    block_start = pid * BLOCK_SIZE

    offs = block_start + tl.arange(0, BLOCK_SIZE)

    mask = offs < num_el

    values = tl.load(
        in_ptr+offs, mask=mask, other=-float("inf")
    )

    block_max = tl.max(
        values, axis=0
    )

    tl.store(part_ptr+pid, block_max)


def triton_max(x):
    assert x.is_cuda
    assert x.dtype == torch.float32
    assert x.ndim==1
    assert x.is_contiguous()

    if x.numel()==0:
        return torch.zeros(1, device=x.device, dtype=x.dtype)

    curr = x
    block_size=1024

    while curr.numel()>1:
        num_el = curr.numel()

        num_programs = triton.cdiv(
            num_el, block_size
        )

        partial = torch.empty(
            num_programs,
            device=x.device,
            dtype=x.dtype
        )

        grid = (num_programs,)

        max_k[grid](
            curr,
            partial,
            num_el,
            BLOCK_SIZE=block_size,
            num_warps=8,
        )

        curr = partial

    return curr

In [28]:
sizes = [
    1,
    7,
    31,
    32,
    255,
    256,
    257,
    1023,
    1024,
    1025,
    100_003,
    1_000_003,
]


for n in sizes:
    x = torch.randn(
        n,
        device="cuda",
        dtype=torch.float32,
    )

    torch.testing.assert_close(
        triton_sum(x).squeeze(),
        torch.sum(x),
        rtol=1e-4,
        atol=1e-3,
    )

    torch.testing.assert_close(
        triton_max(x).squeeze(),
        torch.max(x),
        rtol=0,
        atol=0,
    )

    print(f"n={n}: passed")

n=1: passed
n=7: passed
n=31: passed
n=32: passed
n=255: passed
n=256: passed
n=257: passed
n=1023: passed
n=1024: passed
n=1025: passed
n=100003: passed
n=1000003: passed


In [29]:
x = -torch.rand(
    100_003,
    device="cuda",
    dtype=torch.float32,
)

torch.testing.assert_close(
    triton_max(x).squeeze(),
    torch.max(x),
    rtol=0,
    atol=0,
)